## Preparing Input Data 
The first step in the GalSyn workflow is the preparation of input simulation data. While GalSyn is designed to be agnostic to the specific hydrodynamical simulation used, it requires the raw output to be converted into a standardized HDF5 file format. This ensures that the core routines operate on a consistent data structure regardless of the hydrodynamical simulation data used.  

This example notebook will help guide you through the process of using snapshot and halo data from Swift, specifically using the Eagle subgrid model. The functions involved assume the user has access to Swift snapshots, as well as the VelociRaptor halo finder data. The Velociraptor files are typically stored in a 'halos' subfolder. The notebook outlines one method of selecting a target halo and how to generate the standardised HDF5 file for subsequent analysis.


### Step 1: Defining simulation paths and target halo

Here we define the path to the halo, as well as snapshot name and number. We also establish the target halo number, though this is an optional step.

In [ ]:
import numpy as np
import h5py

# Define the simulation path and snapshot name and number
sim_path    = './Path/To/Your/Simulation/output/'          # Update this to the path where your simulation data is stored
snap_number = 30                                           # Update this to the snapshot number you want to analyze
sim_snap    = f'/snap_{str(snap_number).zfill(4)}.hdf5'     # Check that this is the correct naming convention for your snapshot files

# Optional: Select a halo of a specified mass to extract
target_halo_mass = 1e12 / 1e10  # selecting a roughly milky way sized halo, in units of 1e10 solar masses

# Get the (sub)halo information
halo_properties = sim_path + f'/halos/snap_{str(snap_number).zfill(4)}.VELOCIraptor.properties.0'
halo_masses     = h5py.File(halo_properties, 'r')['Mvir'][:]  # in units of 1e10 solar masses

target_halo_number = np.argmin(np.abs(halo_masses - target_halo_mass))[0]

'cutout_shalo_39_107965.hdf5'

### Step 2: Standardizing the Data Format

In this step creates the standardized GalSyn input format. The function performs necessary unit conversions and identifies the required physical parameters for the next synthesis process.

In [ ]:
from galsyn.simutils_swift import make_sim_file_from_swift_data

# Read the simulation redshift from the cosmology dataset
sim_redshift = h5py.File(sim_path+'/'+sim_snap, 'r')['Cosmology'].attrs['Redshift'][0]

# Define the output path for the standardized file
sim_file = f'./sim_file_swift_{int(snap_number)}_{int(target_halo_number)}.hdf5'

# Run the calculation, the function returns the path to the newly created HDF5 file containing the extracted halo data in a standardized format
converted_file = make_sim_file_from_swift_data(sim_path, sim_snap, target_halo_number=target_halo_number, output_hdf5=sim_file)

Redshift: 1.531239
HDF5 file 'sim_file_tng_39_107965.hdf5' created successfully.
